In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import pysr
from pysr import PySRRegressor

# --- Generate synthetic "climate-like" data ---
np.random.seed(30)
N = 5000

# Inputs
rh = np.random.uniform(0, 1, N)                        # relative humidity
dzrh = np.random.normal(0, 0.01, N)                  # vertical gradient of RH
T = np.random.uniform(250, 300, N)                     # temperature (K)
qc = np.random.exponential(1e-3, N)                # cloud water (kg/kg)
qi = np.random.exponential(5e-4, N)                  # cloud ice (kg/kg)

# Toy "true" relationship for cloud cover (bounded 0–1)
cloud_cover = np.clip(
    0.6*rh - 5*dzrh + 0.002*(280 - T) + 20*qc + 15*qi,
    0, 1
)

# Build DataFrame
data = pd.DataFrame({
    "rh": rh,
    "dzrh": dzrh,
    "T": T,
    "qc": qc,
    "qi": qi,
    "cloud_cover": cloud_cover,
})

# --- Train/test split ---
X = data[["rh", "dzrh", "T", "qc", "qi"]]
y = data["cloud_cover"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Symbolic Regression (PySR) ---
model = PySRRegressor(
    niterations=50,  # keep small for demo
    unary_operators=["exp", "log", "sqrt"],
    binary_operators=["+", "-", "*", "/"],
    loss="loss(x, y) = (x - y)^2"
)
model.fit(X_train, y_train)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/home/b/b309170/my_work/Miniconda3/envs/pysr/lib/python3.10/site-packages/pysr/sr.py:1036: FutureWarning: `loss` has been renamed to `elementwise_loss` in PySRRegressor. Please use that instead.
  warnings.warn(
/home/b/b309170/my_work/Miniconda3/envs/pysr/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
[ Info: Started!



Expressions evaluated per second: 2.420e+03
Progress: 15 / 1550 total iterations (0.968%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.370e+00  0.000e+00  y = 2.4212
2           3.526e-02  4.820e+00  y = exp(-0.98815)
3           3.410e-02  3.353e-02  y = qc + 0.33787
4           2.239e-02  4.207e-01  y = rh * sqrt(0.79479)
5           3.986e-03  1.726e+00  y = (qc + 0.6564) * rh
7           3.239e-03  1.038e-01  y = ((rh * 0.70521) - dzrh) - dzrh
11          2.271e-03  8.875e-02  y = ((rh * ((rh * -0.1213) + 0.74806)) - dzrh) - dzrh
15          1.917e-03  4.240e-02  y = ((dzrh * -1.6793) + (((rh * 2.735) / sqrt(sqrt(T))) - ...
                                      dzrh)) + qc
16          1.596e-03  1.830e-01  y = (sqrt(qc) + rh) * ((((dzrh * -6.326) - qc) - -0.6308) ...
    

[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           3.414e-02  0.000e+00  y = 0.33886
3           4.006e-03  1.071e+00  y = rh * 0.65742
5           3.132e-03  1.230e-01  y = (rh * 0.65711) - dzrh
7           1.580e-03  3.421e-01  y = ((dzrh * 7.6176) - rh) * -0.65587
9           1.241e-03  1.206e-01  y = ((rh * 0.60114) + 0.036658) - (dzrh * 4.9703)
10          9.363e-04  2.821e-01  y = sqrt(qc) + ((rh * 0.61374) - (dzrh * 4.9741))
12          8.870e-04  2.704e-02  y = ((rh * 0.60169) + sqrt(qi + qc)) - (dzrh * 4.9737)
13          4.618e-04  6.528e-01  y = (dzrh * -4.9303) + (((rh * 0.59895) - (T * 0.0019549))...
                                       - -0.57616)
15          3.329e-04  1.636e-01  y = ((rh - (-1.1094 - (-8.9978 * (dzrh - qc)))) * 0.59018)...
                                       - (T * 0.0022297)
16          1.261e-04  9.710e-01  y = (rh * 0.59743) - ((-0.55033 - sqr

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                         0.33886355   
	1         1.071378                                    rh * 0.65741616   
	2         0.123047                            (rh * 0.6571084) - dzrh   
	3         0.342100             ((dzrh * 7.617568) - rh) * -0.65587324   
	4         0.120596  ((rh * 0.60114014) + 0.036658023) - (dzrh * 4....   
	5         0.282110  sqrt(qc) + ((rh * 0.61374056) - (dzrh * 4.9740...   
	6         0.027044  ((rh * 0.6016884) + sqrt(qi + qc)) - (dzrh * 4...   
	7         0.652819  (dzrh * -4.930332) + (((rh * 0.5989481) - (T *...   
	8         0.163563  ((rh - (-1.10942 - (-8.997818 * (dzrh - qc))))...   
	9         0.971041  (rh * 0.5974272) - ((-0.55032676 - sqrt(qc)) -...   
	10        0.042693  (rh * 0.5972154) - ((-0.55268687 - sqrt(qc)) -...   
	11        0.425436  (((rh * 0.59758246) + (dzrh + (qc / 0.05112929...   
	12        0.291449  (dzrh + (rh * 0.5978423)) + ((qc / 0.05100497)...   
	13  >>>>  0.259312  (((qc / 0.051274788) + (rh * 0.5975769)) - (((...   
	14        0.026899  (rh * 0.597583) + ((qc / 0.051235825) + (dzrh ...   
	15        0.017507  (((rh * 0.59758407) + qi) - (((0.09554321 - ((...   
	
	        loss  complexity  
	0   0.034143           1  
	1   0.004006           3  
	2   0.003132           5  
	3   0.001580           7  
	4   0.001241           9  
	5   0.000936          10  
	6   0.000887          12  
	7   0.000462          13  
	8   0.000333          15  
	9   0.000126          16  
	10  0.000116          18  
	11  0.000076          19  
	12  0.000042          21  
	13  0.000025          23  
	14  0.000024          25  
	15  0.000023          27  
]

  - outputs/20250922_142121_F9b6xw/hall_of_fame.csv


In [2]:
# --- Evaluate ---
preds = model.predict(X_test)
print("R²:", r2_score(y_test, preds))

# Extract best equation from PySR (just the formula)
best_equation = model.get_best()["equation"]
print("Best PySR equation:", best_equation)

# Print the original "true" synthetic equation (LaTeX-friendly)
true_equation = (
    r"\mathrm{CloudCover} = "
    r"\mathrm{clip}\Big(0.6 \cdot RH - 5 \cdot \frac{dRH}{dz} "
    r"+ 0.002 \cdot (280 - T) + 20 \cdot q_{cloud} + 15 \cdot q_{ice}, 0, 1\Big)"
)
print("Original equation (LaTeX):", true_equation)

from sympy import Symbol
from IPython.display import display, Math

# display(Math("Best\\ PySR:\\ " + str(best_equation)))
# display(Math("True:\\ " + true_equation))

R²: 0.9991658074987473
Best PySR equation: (((qc / 0.051274788) + (rh * 0.5975769)) - (((0.09557005 - ((dzrh - qi) - qi)) * -5.784529) - (T * -0.0019597588))) + dzrh
Original equation (LaTeX): \mathrm{CloudCover} = \mathrm{clip}\Big(0.6 \cdot RH - 5 \cdot \frac{dRH}{dz} + 0.002 \cdot (280 - T) + 20 \cdot q_{cloud} + 15 \cdot q_{ice}, 0, 1\Big)
